<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Hebbian_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hebbian Learning on FashionMNIST

This notebook implements a simple Hebbian learning network and evaluates its performance on the FashionMNIST dataset.

Unlike conventional neural networks trained using backpropagation, Hebbian learning updates its weights using only local information. The learning rule is inspired by the neuroscience principle often summarized as "neurons that fire together, wire together."

The objective of this notebook is to:

Load and preprocess the FashionMNIST dataset.
Implement a simple Hebbian classifier.
Train the network using a local Hebbian update rule.
Evaluate the model's training time and classification accuracy.
Establish a baseline for future comparisons with more advanced Hebbian and diffusion-based models.

# Importing the Required Libraries

The notebook begins by importing the libraries used throughout the experiment.

PyTorch provides tensor operations and neural network utilities.
Torchvision supplies the FashionMNIST dataset and image transformations.
NumPy offers numerical computing utilities.
time is used to measure training performance.

These libraries provide everything needed to build, train, and evaluate the Hebbian learning model.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
import time

# Preparing the Dataset

Machine learning models require a dataset consisting of input examples and their corresponding labels.

In this experiment we use FashionMNIST, which contains 70,000 grayscale images of clothing items distributed across ten categories, including shirts, shoes, coats, trousers, and bags.

Each image is:

converted into a PyTorch image object,
transformed into a floating-point tensor,
normalized into values between 0 and 1.

The dataset is divided into separate training and testing sets to evaluate how well the model generalizes to previously unseen data.

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]),
)

# Creating DataLoaders

PyTorch's DataLoader simplifies the process of feeding data into the learning algorithm.

During training:

images are grouped into batches of 64,
batches are shuffled each epoch to improve learning,
data is loaded efficiently during training.

The testing DataLoader does not shuffle the data because the order of evaluation does not affect the final accuracy.

In [ ]:
batch_size = 64

train_dataloader = DataLoader(
    training_data,
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

# Verifying the Dataset

Before training a model, it is good practice to inspect the shape of the data.

For FashionMNIST we expect:

Images: (64, 1, 28, 28)
Labels: (64,)

This confirms that each batch contains 64 grayscale images with dimensions 28 x 28 pixels together with their class labels.

In [ ]:
# Data Check
for X, y in train_dataloader:
    print(X.shape)
    print(y.shape)
    break

torch.Size([64, 1, 28, 28])
torch.Size([64])


Building a Primitive Hebbian Network

Instead of using backpropagation, this notebook implements a simple network trained entirely with Hebbian learning.

The model consists of a single weight matrix connecting every input pixel to every output class.

For each training example, the weights are strengthened according to the Hebbian update rule

ΔW=ηxy^T

where

x is the input vector,

y is the desired output,

η is the learning rate.

This local learning rule requires only the activities of connected neurons and does not require gradients or error backpropagation.

In [ ]:
class HebbianNetwork(nn.Module):

    def __init__(self, input_size, output_size, learning_rate=0.01):
        super().__init__()

        self.weights = torch.rand(input_size, output_size)
        self.learning_rate = learning_rate

    def train_hebbian(self, inputs, outputs):
        for x, y in zip(inputs, outputs):
            update = torch.outer(x, y)
            self.weights += self.learning_rate * update

    def train_hebbian_batch(self, inputs, outputs):
        # Batch Hebbian update
        self.weights += self.learning_rate * torch.matmul(
            inputs.T,
            outputs
        )

    def predict(self, x):
        return torch.matmul(x, self.weights)

# Converting Labels into One-Hot Vectors

The FashionMNIST dataset stores labels as integers between 0 and 9.

Hebbian learning updates the weights using output activation vectors rather than integer labels.

Therefore, each label is converted into a one-hot encoded vector, where the correct class is represented by a 1 and every other class is represented by 0.

This representation allows the Hebbian update to strengthen the connections associated with the correct class.

In [ ]:
def one_hot(labels, num_classes=10):
    return torch.eye(num_classes)[labels]

# Evaluating the Hebbian Network

To understand how well the model performs, we define a function that both trains and evaluates the network.

The evaluation consists of two phases.

# Training

For each mini-batch,

flatten the images into vectors,

convert labels into one-hot vectors,

apply the Hebbian learning rule,

repeat for the specified number of epochs.

# Testing

After training,

predictions are computed,

the predicted class is chosen using the maximum activation,

predictions are compared with the true labels.


The function reports two metrics:

Training time measures computational efficiency.

Classification accuracy measures predictive performance.

These metrics provide a simple baseline that can later be compared with more sophisticated learning algorithms.

In [ ]:
def evaluate_network(network, train_dataloader, test_dataloader, epochs):
    start_time = time.time()
    # Training phase
    for epoch in range(epochs):
        for X, y in train_dataloader:
            # Flatten images
            X = X.view(X.size(0), -1)
            # Convert labels to one-hot
            y_onehot = torch.eye(10)[y]
            # Hebbian learning update
            network.train_hebbian_batch( X, y_onehot )
        print(f"Epoch {epoch+1}/{epochs} complete")

    end_time = time.time()
    training_time = end_time - start_time

    # Testing phase
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in test_dataloader:
            # Flatten images
            X = X.view(X.size(0), -1)
            # Get predictions
            predictions = network.predict(X)
            # Convert predictions to class labels
            predicted_classes = torch.argmax( predictions, dim=1)

            # Compare predictions with true labels
            correct += ( predicted_classes == y).sum().item()
            total += y.size(0)

    accuracy = correct / total
    return training_time, accuracy

# Initializing the Hebbian Model

The network is created with three hyperparameters.

Input size = 784

Each FashionMNIST image contains 28 x 28 = 784 pixels.

Output size = 10

There are ten clothing categories.

Learning rate = 0.001

Determines how much the weights change after each Hebbian update.

These parameters define the size of the weight matrix that connects every pixel to every output neuron.

In [ ]:
model = HebbianNetwork(
    input_size=784,
    output_size=10,
    learning_rate=0.001
)

# Training and Testing the Model

The network is trained for ten epochs using the Hebbian learning rule.

After training, two performance metrics are displayed:

Total training time
Classification accuracy on the FashionMNIST test set

These results establish a baseline for evaluating the effectiveness of this simple Hebbian classifier.

In [ ]:
training_time, accuracy = evaluate_network(
    model,
    train_dataloader,
    test_dataloader,
    epochs=10
)

print(f"Training time: {training_time:.2f} seconds")
print(f"Accuracy: {accuracy*100:.2f}%")

Epoch 1/10 complete
Epoch 2/10 complete
Epoch 3/10 complete
Epoch 4/10 complete
Epoch 5/10 complete
Epoch 6/10 complete
Epoch 7/10 complete
Epoch 8/10 complete
Epoch 9/10 complete
Epoch 10/10 complete
Training time: 95.99 seconds
Accuracy: 30.34%


# Discussion of Results

The primitive Hebbian network achieves approximately 30% classification accuracy on the FashionMNIST dataset after ten training epochs.

Although this performance is significantly lower than modern deep neural networks, it demonstrates that a purely local learning rule can extract useful information from image data without using backpropagation or gradient descent.

Several factors contribute to the limited performance:

The network contains only a single linear layer with no hidden representations.
The Hebbian update rule does not explicitly minimize a classification loss.
No normalization or competitive mechanisms are included to prevent uncontrolled weight growth.
The model lacks nonlinear feature extraction, making it difficult to distinguish visually similar clothing categories.

Despite these limitations, this implementation serves as an important baseline. Future iterations can incorporate normalization techniques, multilayer architectures, competitive learning, or diffusion-inspired objectives to investigate whether local learning rules can approach the performance of gradient-based methods while retaining their biological plausibility and computational efficiency.